# Assignment 4: Predicting PM2.5 air quality from meteorology with random forests

Fine particulate matter (**PM2.5**) is the air pollutant responsible for the largest
global health burden. It penetrates deep into the lungs and enters the bloodstream, and
long-term exposure is linked to cardiovascular and respiratory disease. Regulatory networks
measure it directly, but direct measurement is expensive and sparse, so a recurring question
in air quality science is how much of the day-to-day variation can be explained by
**meteorology** and by other, more cheaply measured pollutants.

That is the question you will attack here, with decision trees and random forests. Pollution
concentrations are set by the balance between emissions and the atmosphere's ability to
disperse them, and the dispersion side is meteorological: wind ventilates, a deep mixing
layer dilutes, humidity drives secondary particle formation, temperature controls both
chemistry and the stability of the boundary layer. These relationships are strongly
**nonlinear** and full of **interactions**: which is exactly what trees are good at and
what a linear model handles badly.

You will use **real measurements from the US EPA Air Quality System** for 2023, the year
of the record Canadian wildfire season. Every number in this notebook is an instrument
reading from a regulatory monitor.

Answer each numbered question in the empty cell below it.

In [ ]:
import io
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Load the data

EPA publishes pre-generated annual files, one per measured parameter, with no API key
required. We need six of them: PM2.5 (our target), plus temperature, wind, relative
humidity, ozone and nitrogen dioxide as predictors. Together they are about 27 MB.

Different parameters are measured at different monitoring sites, so the files cannot be
joined at site level without discarding most of the data. We aggregate each one to a
**county-day** mean and join on that. The resulting unit of analysis is "one county on one
day".

In [ ]:
BASE = "https://aqs.epa.gov/aqsweb/airdata/"

def load_epa(filename, usecols):
    """Download one EPA annual file and return it as a dataframe."""
    with urllib.request.urlopen(BASE + filename + ".zip") as resp:
        archive = zipfile.ZipFile(io.BytesIO(resp.read()))
    df = pd.read_csv(archive.open(archive.namelist()[0]), usecols=usecols, low_memory=False)
    df["Date Local"] = pd.to_datetime(df["Date Local"])
    return df

KEY = ["State Code", "County Code", "Date Local"]

# PM2.5 -- the target. We keep the AQI column too; it is the official index value.
pm = load_epa("daily_88101_2023",
              KEY + ["Arithmetic Mean", "AQI", "Latitude", "Longitude",
                     "State Name", "County Name"])
print(f"{len(pm):,} PM2.5 site-days")

data = (pm.groupby(KEY)
          .agg(pm25=("Arithmetic Mean", "mean"), AQI=("AQI", "max"),
               lat=("Latitude", "mean"), lon=("Longitude", "mean"),
               state=("State Name", "first"), county=("County Name", "first"))
          .reset_index())
print(f"{len(data):,} PM2.5 county-days")

# Predictors, each aggregated to the county-day mean and joined on.
for filename, column in [("daily_TEMP_2023", "temp"),
                         ("daily_WIND_2023", "wind"),
                         ("daily_RH_DP_2023", "rh"),
                         ("daily_44201_2023", "o3"),
                         ("daily_42602_2023", "no2")]:
    d = load_epa(filename, KEY + ["Arithmetic Mean"])
    d = d.groupby(KEY)["Arithmetic Mean"].mean().reset_index().rename(
        columns={"Arithmetic Mean": column})
    data = data.merge(d, on=KEY, how="inner")
    print(f"  after joining {column:<5} {len(data):>7,} rows")

data["month"] = data["Date Local"].dt.month
data.head()

The columns are:

| Column | Meaning | Units |
| --- | --- | --- |
| `pm25` | daily mean PM2.5: **the regression target** | µg/m³ |
| `AQI` | official Air Quality Index for PM2.5 | index |
| `temp` | daily mean temperature | °F |
| `wind` | daily mean wind speed | knots |
| `rh` | daily mean relative humidity | % |
| `o3` | daily mean ozone | ppm |
| `no2` | daily mean nitrogen dioxide | ppb |
| `lat`, `lon` | county monitor location | degrees |
| `month` | calendar month | 1–12 |

```{admonition} A note on what this model can and cannot be
:class: note
Ozone and NO2 are *pollutants*, not meteorology. They are included because they carry
information about the state of the atmosphere and about local emissions, and because they
are measured on the same network. But be careful about what that means: a model using NO2
to predict PM2.5 is not a physical model of particle formation, it is exploiting the fact
that both respond to traffic emissions and to the same dispersion conditions. Keep that in
mind when you interpret the feature importances in Part 5.
```

## Part 1: Data exploration

1. How many county-days are in the merged dataset, what date range does it cover, and how many distinct states are represented?

2. Check whether there are any NaNs in the dataframe. If there are, say which columns they are in and drop those rows.

3. Make a histogram of each of the numerical variables. Comment on the shape of the `pm25` distribution in particular, is it symmetric?

4. There are two targets. The numerical one is `pm25`, the daily mean concentration. The
categorical one is the **AQI category**, the band the EPA reports to the public, which is
derived from the AQI value by these breakpoints:

| Category | AQI range | Label |
| --- | --- | --- |
| 0 | 0–50 | Good |
| 1 | 51–100 | Moderate |
| 2 | 101–150 | Unhealthy for Sensitive Groups |
| 3 | 151–200 | Unhealthy |
| 4 | 201+ | Very Unhealthy or worse |

Create a NumPy array `y_regression` containing `pm25`, and a NumPy array
`y_classification` containing the AQI category computed from the `AQI` column.

5. Check how balanced the five classes are. Report the count and the percentage in each.

6. Create a NumPy array called `features` containing these eight predictors:
   - temp
   - wind
   - rh
   - o3
   - no2
   - month
   - lat
   - lon

## Part 2: Preprocessing

7. Create two Python lists: `classnames`, holding the five category labels in the order
given in question 4, and `featurenames`, holding the eight feature names in the order you
built them.

8. Use `StandardScaler` to scale the `features` array, and save the result as a NumPy array `X`.

## Part 3: Training, validation, and test split

9. Split the data into training, validation and test sets, with 80% for training and 10%
each for validation and testing. Carry both targets through the split. Use
`stratify=` on the classification target so the rare categories are represented in every
split, and explain in a comment why that matters here.

## Part 4: Train a random forest classifier

10. Train a `RandomForestClassifier` with 120 estimators and a maximum depth of 10. Set
`class_weight="balanced"`, since the classes are severely imbalanced. Defaults are fine for
the other hyperparameters.

11. Plot the confusion matrix for the trained classifier on the **validation** set as a labelled heatmap, using `classnames` on both axes. Also print the classification report.

The confusion matrix will show that the classifier does poorly on the categories that are
barely represented. One standard response is to over-sample the minority classes. Using the
`imbalanced-learn` library, the **SMOTE** algorithm
[(Chawla et al. 2002)](https://arxiv.org/pdf/1106.1813) synthesizes new minority-class
examples by interpolating between real ones.

In [ ]:
!pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
X_resampled, y_resampled = SMOTE(k_neighbors=3, random_state=0).fit_resample(
    X_train, y_classification_train)

12. Train a second random forest on `X_resampled` and `y_resampled`, using the same hyperparameters as before.

13. Plot the confusion matrix for this second classifier on the validation set, again labelled with `classnames`. Compare it against the first: which classes improved, which got worse, and what happened to overall accuracy?

## Part 5: Train a random forest regressor

14. Train a `RandomForestRegressor` on the training set using the regression target, with 120 estimators and a maximum depth of 10.

15. Predict PM2.5 for the validation set and compute the coefficient of determination ($R^2$) between the true and predicted values. Also report the RMSE in µg/m³, and plot predicted against observed with a 1:1 line.

16. Make a bar plot of the feature importances of the trained regressor, with the x-axis labelled using `featurenames`.

17. Which are the four most important features for predicting PM2.5? Print them with their importance values.

## Part 6: Interpretation

18. Your $R^2$ is likely to be far below what you might have expected, somewhere around
0.3–0.4 rather than the 0.9 that tutorial datasets tend to produce. This is the normal
situation for real environmental prediction, and it is worth understanding rather than
explaining away.

In three or four sentences, say what is **missing** from this feature set that would be
needed to predict PM2.5 well. Think about what actually determines the concentration of
particles over a county on a given day, and what none of these eight variables tells you
about.

*Write your answer here.*

19. Every split in this notebook was random, so days from July can land in the training set
while their immediate neighbours land in the test set. Given that PM2.5 is strongly
autocorrelated from day to day and strongly seasonal, what is wrong with that, and what
split would give an honest estimate of how this model would perform if deployed to fill in
missing monitor data next year?

*Write your answer here.*

20. `lat` and `lon` come out as useful features. Explain what the model is actually
learning from them, and give one reason this should make you cautious about applying the
fitted model to a county that was not in the training data.

*Write your answer here.*